In [38]:
import torch
import torch.nn as nn

1. Простой пример BACKPROPAGATION

In [39]:
# Создаем вычислительный граф "руками"
x = torch.tensor(2.0, requires_grad=True)  # Входные данные
w = torch.tensor(3.0, requires_grad=True)  # Вес
b = torch.tensor(1.0, requires_grad=True)  # Смещение

In [40]:
# Прямой проход: y = w*x + b (линейная функция активации)
y = w * x + b
print(f"Прямой проход: y = {w.item()} * {x.item()} + {b.item()} = {y.item()}")

Прямой проход: y = 3.0 * 2.0 + 1.0 = 7.0


In [41]:
# Предположим, у нас есть целевое значение
target = torch.tensor(10.0)

# Вычисляем ошибку (loss)
loss = (y - target) ** 2
print(f"Ошибка (loss): ({y.item()} - {target.item()})^2 = {loss.item()}")

Ошибка (loss): (7.0 - 10.0)^2 = 9.0


In [42]:
# Обратный проход (backpropagation)
loss.backward()
# Градиенты вычисляются автоматически с помощью .grad!
print(f"∂loss/∂w = {w.grad.item()}")
print(f"∂loss/∂x = {x.grad.item()}")
print(f"∂loss/∂b = {b.grad.item()}")

∂loss/∂w = -12.0
∂loss/∂x = -18.0
∂loss/∂b = -6.0


In [44]:
# Проверим вычисления вручную:
# loss = (y - 10)^2 = (w*x + b - 10)^2
# ∂loss/∂w = 2*(w*x + b - 10)*x = 2*(6 + 1 - 10)*2 = 2*(-3)*2 = -12
# ∂loss/∂x = 2*(w*x + b - 10)*w = 2*(-3)*3 = -18
# ∂loss/∂b = 2*(w*x + b - 10)*1 = 2*(-3) = -6

Теперь становится ясно  в какую сторону корректировать веса - в противоположную знаку градиента для решения задачи по критерию минимума loss

In [45]:
# поменяем веса вручную с шагом 0.1
h = 0.1
w = w - h * w.grad.item()  # Вес
b = b - h * b.grad.item()  # Смещение

In [46]:
# Еще раз выполним прямой проход: y = w*x + b
y = w * x + b
print(f"Прямой проход: y = {w.item()} * {x.item()} + {b.item()} = {y.item()}")

Прямой проход: y = 4.199999809265137 * 2.0 + 1.600000023841858 = 10.0


In [47]:
# Вычисляем ошибку (loss)
loss = (y - target) ** 2
print(f"Ошибка (loss): ({y.item()} - {target.item()})^2 = {loss.item()}")

Ошибка (loss): (10.0 - 10.0)^2 = 0.0


In [10]:
# уровень ошибк снизился

In [11]:
# поменяем веса резко на 5
w = torch.tensor(8.1, requires_grad=True)  # Вес
b = torch.tensor(6.2, requires_grad=True)  # Смещение

In [12]:
# Еще раз выполним прямой проход: y = w*x + b
y = w * x + b
print(f"Прямой проход: y = {w.item()} * {x.item()} + {b.item()} = {y.item()}")

Прямой проход: y = 8.100000381469727 * 2.0 + 6.199999809265137 = 22.400001525878906


In [13]:
# Вычисляем ошибку (loss)
loss = (y - target) ** 2
print(f"Ошибка (loss): ({y.item()} - {target.item()})^2 = {loss.item()}")

Ошибка (loss): (22.400001525878906 - 10.0)^2 = 153.76004028320312


Вывод: нужно плавное изменение шага обучения

2. Пример с нейросетью

In [48]:
# Создаем простую полносвязную нейронную сеть прямого распространения на PyTorch
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(3, 4)  # 3 входа, 4 нейрона
        self.fc2 = nn.Linear(4, 2)  # 4 входа, 2 выхода - задача классификации
        self.activation = nn.ReLU() # применим к скрытому слою усеченную линейную функцию активации

    def forward(self, x):
        x = self.fc1(x)
        x = self.activation(x)
        x = self.fc2(x)
        return x

In [49]:
model = SimpleNN() # инициализация модели
print(model)

SimpleNN(
  (fc1): Linear(in_features=3, out_features=4, bias=True)
  (fc2): Linear(in_features=4, out_features=2, bias=True)
  (activation): ReLU()
)


In [50]:
# Создаем тестовые данные
inputs = torch.randn(2, 3)  # 2 примера, 3 признака
targets = torch.tensor([[1.0, 0.0], [0.0, 1.0]])  # закодируем таргеты one-hot encoding

In [ ]:
# Индексы положительных классов в каждом списке
torch.argmax(targets, dim=1)

tensor([0, 1])

In [51]:
# Прямой проход
outputs = model(inputs)
print(f"\nВыход сети:\n{outputs}")

# Функция потерь
criterion = nn.CrossEntropyLoss() # в качестве функции потерь берем категориальную кросс-экнтропию, т.к. это задача классификации
loss = criterion(outputs, torch.argmax(targets, dim=1))
print(f"\nФункция потерь: {loss.item()}")


Выход сети:
tensor([[-0.2793, -0.3932],
        [-0.1446, -0.2187]], grad_fn=<AddmmBackward0>)

Функция потерь: 0.6843418478965759


3. РУЧНОЙ BACKPROPAGATION

In [58]:
# Сброс градиентов
model.zero_grad()

# Сохраняем начальные веса
initial_weights = {}
for name, param in model.named_parameters():
    initial_weights[name] = param.data.clone()
    print(f"{name}: форма = {param.shape}")

# Обратный проход
loss.backward()

print("\nГрадиенты после backward():")
for name, param in model.named_parameters(): # итерируемся по параметрам
    if param.grad is not None:
        print(f"{name}:")
        print(f"  Градиенты (первые 5 значений): {param.grad.flatten()[:5]}")
        print(f"  Среднее значение градиента: {param.grad.mean().item():.6f}")

fc1.weight: форма = torch.Size([4, 3])
fc1.bias: форма = torch.Size([4])
fc2.weight: форма = torch.Size([2, 4])
fc2.bias: форма = torch.Size([2])

Градиенты после backward():
fc1.weight:
  Градиенты (первые 5 значений): tensor([0., 0., 0., 0., 0.])
  Среднее значение градиента: 0.014156
fc1.bias:
  Градиенты (первые 5 значений): tensor([ 0.0000,  0.0000, -0.1150,  0.0008])
  Среднее значение градиента: -0.028568
fc2.weight:
  Градиенты (первые 5 значений): tensor([0.0000, 0.0000, 0.0379, 0.2024, 0.0000])
  Среднее значение градиента: 0.000000
fc2.bias:
  Градиенты (первые 5 значений): tensor([ 0.0235, -0.0235])
  Среднее значение градиента: 0.000000


4. ОБНОВЛЕНИЕ ВЕСОВ

In [59]:
# Оптимизатор (использует градиенты для обновления весов)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01) # фиксированный шаг обучения

# Показываем веса до обновления
print("Веса ДО обновления:")
for name, param in model.named_parameters():
    if 'weight' in name:
        print(f"{name}: среднее = {param.data.mean().item():.6f}")

# Шаг оптимизатора (обновление весов)
optimizer.step()

print("\nВеса ПОСЛЕ обновления:")
for name, param in model.named_parameters():
    if 'weight' in name:
        old_mean = initial_weights[name].mean().item()
        new_mean = param.data.mean().item()
        print(f"{name}:")
        print(f"  Было: среднее = {old_mean:.6f}")
        print(f"  Стало: среднее = {new_mean:.6f}")
        print(f"  Изменение: {new_mean - old_mean:.6f}")

Веса ДО обновления:
fc1.weight: среднее = -0.115864
fc2.weight: среднее = 0.121289

Веса ПОСЛЕ обновления:
fc1.weight:
  Было: среднее = -0.115864
  Стало: среднее = -0.116006
  Изменение: -0.000142
fc2.weight:
  Было: среднее = 0.121289
  Стало: среднее = 0.121289
  Изменение: 0.000000


5. ПОЛНЫЙ ЦИКЛ ОБУЧЕНИЯ

In [61]:
# Создаем более простую сеть для демонстрации
class TinyNN(nn.Module):
    def __init__(self):
        super(TinyNN, self).__init__()
        self.layer = nn.Linear(2, 1)

    def forward(self, x):
        return self.layer(x)

# Подготовка данных
# Задача: логическая операция AND таблица истинности:
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = torch.tensor([[0.], [0.], [0.], [1.]])  # AND операция

# Создаем модель, функцию потерь и оптимизатор
tiny_model = TinyNN()
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(tiny_model.parameters(), lr=0.5)

print(f"Исходные веса: {tiny_model.layer.weight.data}")
print(f"Исходное смещение: {tiny_model.layer.bias.data}")

# Цикл обучения
epochs = 100
print("\nОбучение...")
for epoch in range(epochs):
    # Прямой проход
    predictions = tiny_model(X)
    loss = criterion(predictions, y)

    # Обратный проход
    optimizer.zero_grad()  # Обнуляем градиенты
    loss.backward()       # Вычисляем градиенты

    # Обновляем веса
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Эпоха [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Исходные веса: tensor([[-0.5763, -0.1668]])
Исходное смещение: tensor([-0.5713])

Обучение...
Эпоха [10/100], Loss: 0.0667
Эпоха [20/100], Loss: 0.0626
Эпоха [30/100], Loss: 0.0625
Эпоха [40/100], Loss: 0.0625
Эпоха [50/100], Loss: 0.0625
Эпоха [60/100], Loss: 0.0625
Эпоха [70/100], Loss: 0.0625
Эпоха [80/100], Loss: 0.0625
Эпоха [90/100], Loss: 0.0625
Эпоха [100/100], Loss: 0.0625


In [62]:
# Проверка результатов
print("\nРезультаты после обучения:")
with torch.no_grad():  # Отключаем вычисление градиентов для инференса
    test_inputs = torch.tensor([[0., 0.], [1., 1.]])
    predictions = tiny_model(test_inputs)
    print(f"Вход: {test_inputs[0]}, Предсказание: {predictions[0].item():.4f}")
    print(f"Вход: {test_inputs[1]}, Предсказание: {predictions[1].item():.4f}")

print(f"\nФинальные веса: {tiny_model.layer.weight.data}")
print(f"Финальное смещение: {tiny_model.layer.bias.data}")


Результаты после обучения:
Вход: tensor([0., 0.]), Предсказание: -0.2500
Вход: tensor([1., 1.]), Предсказание: 0.7500

Финальные веса: tensor([[0.5000, 0.5000]])
Финальное смещение: tensor([-0.2500])


In [ ]:
# Что нужно сделать чтобы исправить???

In [63]:
class TinyNN(nn.Module):
    def __init__(self):
        super(TinyNN, self).__init__()
        self.layer = nn.Linear(2, 1)

    def forward(self, x):
        return torch.sigmoid(self.layer(x))  # Сигмоида для получения вероятностей

# Данные
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = torch.tensor([[0.], [0.], [0.], [1.]])

# Модель и функция потерь
tiny_model = TinyNN()
criterion = nn.BCELoss()  # Binary Cross Entropy для бинарной классификации
optimizer = torch.optim.SGD(tiny_model.parameters(), lr=1.0)

# Обучение
epochs = 1000
for epoch in range(epochs):
    predictions = tiny_model(X)
    loss = criterion(predictions, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"Эпоха [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

# Проверка
print("\nРезультаты после обучения:")
with torch.no_grad():
    for i in range(4):
        pred = tiny_model(X[i])
        print(f"Вход: {X[i]}, Предсказание: {pred.item():.4f}")

Эпоха [100/1000], Loss: 0.1393
Эпоха [200/1000], Loss: 0.0802
Эпоха [300/1000], Loss: 0.0558
Эпоха [400/1000], Loss: 0.0426
Эпоха [500/1000], Loss: 0.0344
Эпоха [600/1000], Loss: 0.0288
Эпоха [700/1000], Loss: 0.0247
Эпоха [800/1000], Loss: 0.0217
Эпоха [900/1000], Loss: 0.0193
Эпоха [1000/1000], Loss: 0.0174

Результаты после обучения:
Вход: tensor([0., 0.]), Предсказание: 0.0000
Вход: tensor([0., 1.]), Предсказание: 0.0202
Вход: tensor([1., 0.]), Предсказание: 0.0202
Вход: tensor([1., 1.]), Предсказание: 0.9718


6. ВИЗУАЛИЗАЦИЯ ВЫЧИСЛИТЕЛЬНОГО ГРАФА

In [65]:
# Создаем простой граф
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)
c = a * b
d = torch.tensor(4.0, requires_grad=True)
e = c + d
e.backward()

print(f"Узел a: значение={a.item()}, градиент={a.grad.item()}")
print(f"Узел b: значение={b.item()}, градиент={b.grad.item()}")
print(f"Узел c: значение={c.item()}, градиент={c.grad}")  # c.grad is None, так как это промежуточный узел
print(f"Узел d: значение={d.item()}, градиент={d.grad.item()}")
print(f"Узел e: значение={e.item()}, градиент={e.grad}")  # e.grad is None, так как мы вызвали backward() от e

Узел a: значение=2.0, градиент=3.0
Узел b: значение=3.0, градиент=2.0
Узел c: значение=6.0, градиент=None
Узел d: значение=4.0, градиент=1.0
Узел e: значение=10.0, градиент=None


C:\Users\MazurenkoEV\AppData\Local\Temp\ipykernel_7708\1466255524.py:11: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\build\aten\src\ATen/core/TensorBody.h:497.)
  print(f"Узел c: значение={c.item()}, градиент={c.grad}")  # c.grad is None, так как это промежуточный узел
C:\Users\MazurenkoEV\AppData\Local\Temp\ipykernel_7708\1466255524.py:13: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be 

КЛЮЧЕВЫЕ ВЫВОДЫ:
1. requires_grad=True - говорит PyTorch отслеживать операции для вычисления градиентов
2. forward() - вычисляет выход сети
3. backward() - автоматически вычисляет градиенты для всех тензоров с requires_grad=True
4. optimizer.step() - обновляет веса используя вычисленные градиенты
5. optimizer.zero_grad() - очищает градиенты перед следующим backward()
6. Вычислительный граф автоматически создается при выполнении операций
7. Градиенты накапливаются (accumulate), поэтому их нужно обнулять

In [27]:
pip install torchviz

In [66]:
import torchviz
# Создаем простую модель
class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.fc1 = nn.Linear(3, 5)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(5, 2)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.softmax(x)
        return x

model = SimpleModel()

# Создаем тестовые данные
x = torch.randn(1, 3, requires_grad=True)

# Прямой проход
y = model(x)

# Визуализация графа с torchviz
# Сохраняем граф в файл
dot = torchviz.make_dot(y, params=dict(list(model.named_parameters()) + [('x', x)]))

# Сохраняем в разных форматах
dot.render("computational_graph", format="png", cleanup=True)
print("Граф сохранен как computational_graph.png")

# Альтернативный способ с более детальной информацией
dot_detailed = torchviz.make_dot(y,
                                     params=dict(model.named_parameters()),
                                     show_attrs=True,
                                     show_saved=True)
dot_detailed.render("detailed_graph", format="png", cleanup=True)
print("Детальный граф сохранен как detailed_graph.png")


ModuleNotFoundError: No module named 'torchviz'